# Explore the data (lesson 06)

A first look at the inputs before any rules are applied: how many ClinVar
variants per gene and label, how many are in gnomAD, and how their
frequencies are distributed. Exploration only: the results that count are
computed by the workflow and shown in the Quarto report.

Run it headless with `pixi run explore` (it runs the workflow first, so the
tables exist). Before committing, clear outputs: `pixi run clear-notebooks`.

In [ ]:
from pathlib import Path
import os

# Work from the project folder, whether started from notebooks/ or the root
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

clinvar = pd.read_csv("data/processed/clinvar_panel.tsv", sep="\t")
gnomad = pd.read_csv("data/processed/gnomad_panel.tsv", sep="\t", keep_default_na=False)
print(f"ClinVar: {len(clinvar):,} variants, {clinvar.shape[1]} columns")
print(f"gnomAD:  {len(gnomad):,} records in the gene slices")

## ClinVar labels per gene

In [ ]:
clinvar.groupby(["gene", "label"]).size().unstack(fill_value=0)

## Review status and molecular consequence

In [ ]:
display(clinvar["stars"].value_counts().sort_index().rename("variants"))
clinvar["consequence"].value_counts().head(10)

## gnomAD filters

How many gnomAD records pass gnomAD's quality filters (only PASS records are used).

In [ ]:
gnomad["filter"].value_counts()

## Frequencies of ClinVar variants found in gnomAD

In [ ]:
key = ["chrom", "pos", "ref", "alt"]
merged = clinvar.merge(gnomad[gnomad["filter"] == "PASS"], on=key, how="left")
merged["faf"] = pd.to_numeric(merged["faf"].replace(".", None))
found = merged.groupby("label")["faf"].apply(lambda s: s.notna().mean()).rename("share in gnomAD")
found

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
for label, color in [("benign", "#2a78d6"), ("pathogenic", "#eb6834")]:
    values = merged.loc[(merged["label"] == label) & (merged["faf"] > 0), "faf"]
    ax.hist(values, bins=pd.Series([10 ** (x / 4) for x in range(-32, 1)]), color=color,
            alpha=0.7, label=f"ClinVar {label} (n={len(values)})")
ax.set_xscale("log")
ax.set_xlabel("gnomAD group-max FAF")
ax.set_ylabel("Variants")
ax.legend(frameon=False, fontsize=8)

# Show the figure as an <img> with alternative text (plt.show() gives none)
import base64
import io
from IPython.display import HTML

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
plt.close(fig)
n_benign = int(((merged["label"] == "benign") & (merged["faf"] > 0)).sum())
n_pathogenic = int(((merged["label"] == "pathogenic") & (merged["faf"] > 0)).sum())
alt = (
    "Overlaid histograms of the gnomAD group-max filtering allele frequency "
    "(log scale, 1e-8 to 1) for ClinVar variants found in gnomAD: "
    f"{n_benign} benign or likely benign (blue) and {n_pathogenic} pathogenic "
    "or likely pathogenic (orange). Benign variants reach much higher "
    "frequencies than pathogenic ones."
)
png = base64.b64encode(buf.getvalue()).decode()
HTML(f'<img src="data:image/png;base64,{png}" alt="{alt}">')

## Notes for the report

- Benign variants sit at much higher frequencies than pathogenic ones, as expected.
- Many pathogenic variants are absent from gnomAD: PM2 applies to them with or without frequencies.
- Kept for the report: labels per gene (table), frequency evidence per label and scenario (figure), threshold sweep.